# HEFT/HEFT-DS Imitation MLP — reward-weighted supervised learning

**Input:** `sim_res/training/*.csv` produced by `BatchTrainingRunner` (HEFT + HEFT-DS).
Each row is a scheduling decision: 50 features describing the simulator state +
the chosen node index (label 0–19).

**Output:** `simulator/src/main/resources/models/heft_imitation.onnx` — a 50→128→64→20
MLP that can be loaded by `NeuralScheduler.java` via ONNX Runtime.

**Trick:** *reward-weighted* loss — every decision is weighted by `1/makespan_seconds`
of the run it came from, so the network preferentially imitates decisions taken
during *fast* runs, not all decisions equally.

In [1]:
import os, json, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

REPO_ROOT = Path('..').resolve()          # simulator/
DATA_DIR  = REPO_ROOT / 'sim_res' / 'training'
MODEL_OUT = REPO_ROOT / 'src' / 'main' / 'resources' / 'models' / 'heft_imitation.onnx'
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)

DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {DEVICE}')
print(f'data:   {DATA_DIR}')
print(f'output: {MODEL_OUT}')

device: mps
data:   /Users/busdavid/szakdoga/DISSECT-CF-Fog/simulator/sim_res/training
output: /Users/busdavid/szakdoga/DISSECT-CF-Fog/simulator/src/main/resources/models/heft_imitation.onnx


## 1. Load CSVs + JSON sidecars

For every CSV we look up its sidecar JSON to get `makespan_seconds` — that's the
reward signal. Rows from the same run share the same weight.

In [ ]:
FEATURE_COLS = [
    'task_runtime', 'task_rank_up', 'task_in_degree', 'task_out_degree',
    'task_total_input_bytes',
    'elapsed_sec', 'tasks_done', 'tasks_remaining', 'total_active_vms', 'avg_node_load',
]
for i in range(20):
    FEATURE_COLS += [f'node{i}_queue', f'node{i}_running']
assert len(FEATURE_COLS) == 50

import re

# Imitate the EFT-greedy teachers only. MaxMin / AdaptiveHEFT-distributed pick
# their node via a workflow-deterministic shuffle that the state features
# cannot predict, so training on their decisions collapses the network onto
# whichever edge node is most common in the shuffle. HEFT + HEFT-DS produce
# state-deterministic labels (lowest-EFT processor), which the network can
# actually learn. The NeuralScheduler at inference time falls back to
# AdaptiveHeftScheduler (its parent), which handles the workflows where
# round-robin would have been the right call — so we get the best of both
# without trying to learn the unlearnable.
TEACHER_SCHEDULERS = {'heft', 'heftds'}

def workflow_key(csv_stem: str) -> str:
    return re.sub(r'_(heft|heftds|maxmin|adaptive)_\d+$', '', csv_stem)

rows = []
kept_runs = 0
for csv_path in sorted(DATA_DIR.glob('*.csv')):
    json_path = csv_path.with_suffix('.json')
    if not json_path.exists():
        json_path = Path(str(csv_path)[:-4] + '.meta.json')
    if not json_path.exists():
        continue
    meta = json.loads(json_path.read_text())
    scheduler = meta.get('scheduler', '?')
    if scheduler not in TEACHER_SCHEDULERS:
        continue
    ms  = float(meta['makespan_seconds'])
    kwh = float(meta.get('energy_kwh', 0.0))
    if ms <= 0:
        continue
    df = pd.read_csv(csv_path)
    df['weight']     = 1.0 / ms           # reward = inverse makespan
    df['source_run'] = csv_path.stem
    df['scheduler']  = scheduler
    df['workflow']   = workflow_key(csv_path.stem)
    rows.append(df)
    kept_runs += 1

data = pd.concat(rows, ignore_index=True)
print(f'loaded {kept_runs} EFT-greedy teacher runs '
      f'({", ".join(sorted(TEACHER_SCHEDULERS))}), '
      f'{len(data)} decisions, {data.workflow.nunique()} workflows')
print('decisions per scheduler:')
print(data.groupby('scheduler').size().to_string())
data.head()

## 2. Feature standardization

z-score normalisation; the µ and σ get saved alongside the ONNX so that
`NeuralScheduler.java` can apply the same transform at inference time.

In [ ]:
X = data[FEATURE_COLS].to_numpy(dtype=np.float32)
y = data['chosen_node_idx'].to_numpy(dtype=np.int64)
w = data['weight'].to_numpy(dtype=np.float32)
groups = data['source_run'].to_numpy()

mu  = X.mean(axis=0)
sig = X.std(axis=0)
sig[sig < 1e-6] = 1.0
X = (X - mu) / sig

# ----- class-imbalance compensation -----
# Use class weighting ONLY when the teacher data is naturally skewed (the
# HEFT-only case where cloud nodes dominate). For multi-teacher distillation
# the round-robin teachers (MaxMin, Adaptive HEFT in distributed mode) already
# balance per-node labels, and forcing additional inverse-frequency weighting
# *inverts* the bias and confuses the network. Auto-detect by looking at how
# uneven the natural label distribution is.
NUM_CLASSES = 20
counts = np.bincount(y, minlength=NUM_CLASSES).astype(np.float32)
class_freq = counts / counts.sum()
# Gini-ish skew measure: 1.0 = perfectly balanced, 0.0 = single class
imbalance = class_freq.std() / (class_freq.mean() + 1e-9)
USE_CLASS_WEIGHTS = imbalance > 1.0       # heuristic; HEFT-only had ~1.5, multi-teacher ~0.7
print(f'label-distribution imbalance ratio: {imbalance:.3f}  -> '
      f'class weighting {"ON" if USE_CLASS_WEIGHTS else "OFF"}')

if USE_CLASS_WEIGHTS:
    class_freq_safe = class_freq.copy()
    class_freq_safe[class_freq_safe == 0] = 1.0
    class_weight = 1.0 / class_freq_safe
    class_weight = class_weight / class_weight.mean()
    print('class weights (sample count -> weight):')
    for c in range(NUM_CLASSES):
        print(f'  node{c:2d}  count={int(counts[c]):5d}  weight={class_weight[c]:.3f}')
    w = w * class_weight[y]
else:
    print('label counts per node (no class-reweighting applied):')
    for c in range(NUM_CLASSES):
        print(f'  node{c:2d}  count={int(counts[c]):5d}')

# normalise so mean weight is 1
w = w / w.mean()

print(f'\nX: {X.shape}  y: {y.shape}  w: mean={w.mean():.3f} min={w.min():.3f} max={w.max():.3f}')
print('mu, sig saved alongside model export')

## 3. Train / val split *by run*

Naive row-wise split leaks: decisions from the *same* workflow run are highly
correlated. Splitting by `source_run` keeps an entire run's decisions together,
so val accuracy reflects out-of-distribution workflow generalisation.

In [4]:
rng = np.random.default_rng(42)
unique_runs = np.array(sorted(set(groups)))
rng.shuffle(unique_runs)
n_val = max(1, int(0.2 * len(unique_runs)))
val_runs   = set(unique_runs[:n_val])
train_runs = set(unique_runs[n_val:])

train_idx = np.array([i for i, g in enumerate(groups) if g in train_runs])
val_idx   = np.array([i for i, g in enumerate(groups) if g in val_runs])
print(f'runs: {len(train_runs)} train / {len(val_runs)} val')
print(f'rows: {len(train_idx)} train / {len(val_idx)} val')

runs: 9 train / 2 val
rows: 1186 train / 399 val


## 4. MLP — 50 → 128 → 64 → 20

In [5]:
class HeftMlp(nn.Module):
    def __init__(self, in_dim=50, hidden=(128, 64), out_dim=20, dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, out_dim)]    # logits; softmax applied via loss
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = HeftMlp().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'parameters: {n_params:,}')

HeftMlp(
  (net): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=20, bias=True)
  )
)
parameters: 16,084


## 5. Training — weighted cross-entropy + early stopping

In [6]:
BATCH    = 256
EPOCHS   = 100                   # class weighting makes the problem harder; needs more epochs
LR       = 1e-3
PATIENCE = 8                     # stop if val loss doesn't improve in N epochs

def to_tensor(idx):
    return (
        torch.from_numpy(X[idx]).float(),
        torch.from_numpy(y[idx]).long(),
        torch.from_numpy(w[idx]).float(),
    )

Xtr, ytr, wtr = to_tensor(train_idx)
Xva, yva, wva = to_tensor(val_idx)

train_loader = DataLoader(TensorDataset(Xtr, ytr, wtr), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xva, yva, wva), batch_size=BATCH)

opt = torch.optim.Adam(model.parameters(), lr=LR)
ce  = nn.CrossEntropyLoss(reduction='none')     # per-sample, so we can weight

best_val = float('inf')
best_state = None
stale = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0; tr_correct = 0; tr_n = 0
    for xb, yb, wb in train_loader:
        xb, yb, wb = xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
        opt.zero_grad()
        logits = model(xb)
        loss = (ce(logits, yb) * wb).mean()
        loss.backward(); opt.step()
        tr_loss    += loss.item() * len(xb)
        tr_correct += (logits.argmax(1) == yb).sum().item()
        tr_n       += len(xb)
    tr_loss /= tr_n; tr_acc = tr_correct / tr_n

    model.eval()
    va_loss = 0.0; va_correct = 0; va_n = 0
    with torch.no_grad():
        for xb, yb, wb in val_loader:
            xb, yb, wb = xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
            logits = model(xb)
            loss = (ce(logits, yb) * wb).mean()
            va_loss    += loss.item() * len(xb)
            va_correct += (logits.argmax(1) == yb).sum().item()
            va_n       += len(xb)
    va_loss /= va_n; va_acc = va_correct / va_n

    flag = ''
    if va_loss < best_val - 1e-4:
        best_val   = va_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0; flag = ' ✓'
    else:
        stale += 1
    print(f'epoch {epoch:2d}  train loss={tr_loss:.4f} acc={tr_acc:.3f}   val loss={va_loss:.4f} acc={va_acc:.3f}{flag}')
    if stale >= PATIENCE:
        print(f'early stop @ epoch {epoch} (no val improvement in {PATIENCE} epochs)')
        break

model.load_state_dict(best_state)
print(f'best val loss: {best_val:.4f}')

epoch  1  train loss=3.4741 acc=0.108   val loss=1.0831 acc=0.128 ✓
epoch  2  train loss=2.7502 acc=0.178   val loss=1.0874 acc=0.080
epoch  3  train loss=2.1418 acc=0.175   val loss=1.0950 acc=0.055
epoch  4  train loss=1.6960 acc=0.149   val loss=1.1077 acc=0.023
epoch  5  train loss=1.4662 acc=0.147   val loss=1.1262 acc=0.018
epoch  6  train loss=1.3463 acc=0.136   val loss=1.1488 acc=0.018
epoch  7  train loss=1.2576 acc=0.139   val loss=1.1706 acc=0.018
epoch  8  train loss=1.1886 acc=0.136   val loss=1.1848 acc=0.023
epoch  9  train loss=1.1487 acc=0.142   val loss=1.1883 acc=0.033
early stop @ epoch 9 (no val improvement in 8 epochs)
best val loss: 1.0831


## 6. Confusion + per-class accuracy on val

Quick diagnostic — which nodes does the model confuse?

In [7]:
from collections import Counter
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb, _ in val_loader:
        logits = model(xb.to(DEVICE))
        all_pred.append(logits.argmax(1).cpu().numpy())
        all_true.append(yb.numpy())
pred = np.concatenate(all_pred); true = np.concatenate(all_true)

print('per-node val accuracy (only nodes that appear in val):')
for n in range(20):
    mask = (true == n)
    if mask.sum() == 0: continue
    acc = (pred[mask] == n).mean()
    print(f'  node{n:2d}  n={mask.sum():4d}  acc={acc:.3f}')

miscls = Counter()
for t, p in zip(true, pred):
    if t != p: miscls[(int(t), int(p))] += 1
print('\ntop 10 confusion pairs (true → pred):')
for (t, p), c in miscls.most_common(10):
    print(f'  node{t:2d} → node{p:2d}   {c}')

per-node val accuracy (only nodes that appear in val):
  node 0  n=  22  acc=0.000
  node 1  n=  11  acc=0.000
  node 2  n=  59  acc=0.000
  node 3  n=  10  acc=0.000
  node 4  n=  55  acc=0.909
  node 5  n=  10  acc=0.000
  node 6  n=  10  acc=0.000
  node 7  n=  53  acc=0.000
  node 8  n=   7  acc=0.000
  node 9  n=   7  acc=0.143
  node10  n=   8  acc=0.000
  node11  n=  10  acc=0.000
  node12  n=   7  acc=0.000
  node13  n=   7  acc=0.000
  node14  n=   7  acc=0.000
  node15  n=  45  acc=0.000
  node16  n=   6  acc=0.000
  node17  n=   4  acc=0.000
  node18  n=  44  acc=0.000
  node19  n=  17  acc=0.000

top 10 confusion pairs (true → pred):
  node 2 → node 4   55
  node 7 → node 4   51
  node15 → node 4   41
  node18 → node 4   41
  node 0 → node 4   22
  node19 → node 4   16
  node 1 → node 4   10
  node 3 → node 4   9
  node 5 → node 4   9
  node 6 → node 4   8


## 7. ONNX export

We export a model that **includes the z-score normalisation** as the first
operation. That way the Java side just feeds raw features and gets logits back,
no need to ship µ/σ separately and risk drift between Python and Java.

In [8]:
class WrappedModel(nn.Module):
    """Inference-time wrapper: standardisation + MLP. Input = raw features."""
    def __init__(self, mlp, mu, sig):
        super().__init__()
        self.mlp = mlp
        self.register_buffer('mu',  torch.from_numpy(mu).float())
        self.register_buffer('sig', torch.from_numpy(sig).float())

    def forward(self, x):
        return self.mlp((x - self.mu) / self.sig)

wrapped = WrappedModel(model, mu, sig).to('cpu').eval()
dummy = torch.zeros(1, 50, dtype=torch.float32)
torch.onnx.export(
    wrapped, dummy, MODEL_OUT.as_posix(),
    input_names=['features'], output_names=['logits'],
    dynamic_axes={'features': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=14,
)
size_kb = MODEL_OUT.stat().st_size / 1024
print(f'✓ exported {MODEL_OUT}  ({size_kb:.1f} KB)')

# Sanity check: re-load the ONNX, run inference, compare to PyTorch output
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(MODEL_OUT.as_posix())
    x_test = X[val_idx[:8]].astype(np.float32) * sig + mu  # un-normalise so input is raw
    onnx_out  = sess.run(['logits'], {'features': x_test})[0]
    torch_out = wrapped(torch.from_numpy(x_test)).detach().numpy()
    max_diff = np.abs(onnx_out - torch_out).max()
    print(f'ONNX vs PyTorch max logit diff: {max_diff:.2e}  (should be < 1e-5)')
except ImportError:
    print('onnxruntime not installed; skipping cross-check (pip install onnxruntime)')

/var/folders/8_/xmwv32ks4p9d8jk8ddcqchfw0000gn/T/ipykernel_70517/1290465817.py:14: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0513 02:31:40.297000 70517 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0513 02:31:40.627000 70517 site-packages/torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0513 02:31:40

[torch.onnx] Obtain model graph for `WrappedModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `WrappedModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).
Failed to convert the model to the target version 14 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/Users/busdavid/Library/Python/3.13/lib/python/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/Users/busdavid/Library/Python/3.13/lib/python/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/Users/busdavid/Library/Pyt

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✓ exported /Users/busdavid/szakdoga/DISSECT-CF-Fog/simulator/src/main/resources/models/heft_imitation.onnx  (11.7 KB)
ONNX vs PyTorch max logit diff: 8.94e-08  (should be < 1e-5)


## Done

Next: implement `NeuralScheduler.java` that loads this `.onnx`, runs it at each
scheduling decision, and falls back to HEFT if the top-1 confidence is below a
threshold (e.g. 0.5).